In [1]:
import math
import numpy as np
from load import load_bin_dir

In [2]:

def BIC(target_bin, control_bin, lambda_ = 2):
    if len(target_bin) != len(control_bin):
        raise Exception('target and control should have same number of bins')

    first_item = 0
    for i in range(len(target_bin)):
        sum_trg_control = target_bin[i]+ control_bin[i]
        if not((target_bin[i] == 0) or (control_bin[i] == 0) or (sum_trg_control == 0)):
            first_item = first_item + (target_bin[i] * math.log10(target_bin[i] / sum_trg_control)) + (control_bin[i] * math.log10(control_bin[i] / sum_trg_control))
        elif sum_trg_control == 0:
            pass
        elif target_bin[i] == 0:
            first_item = first_item + (control_bin[i] * math.log10(control_bin[i] / sum_trg_control))
        elif control_bin[i] == 0:
            first_item = first_item + (target_bin[i] * math.log10(target_bin[i] / sum_trg_control))
    first_item = -2 * first_item 
    sum_of_all_bin = lambda_ * len(target_bin) * math.log10(np.sum(target_bin)+np.sum(control_bin))
    
    return round(first_item + sum_of_all_bin, 4)


In [22]:
def BICseq2_segmentation(target_bin, control_bin, lambda_ = 2):
    if len(target_bin) != len(control_bin):
        raise Exception('target and control should have same number of bins')
    
    ini_BIC = BIC(target_bin, control_bin, lambda_)
    start_bin, end_bin = np.arange(len(target_bin)), np.arange(len(target_bin))

    while(True):
        # if only has one bin then don't need to merge
        if len(target_bin) == 1 :
            break
        
        # calculate each posible merge's BIC in this iteration
        BIC_when_merge = np.zeros(len(target_bin)-1)
        for i in range(len(BIC_when_merge)):
            BIC_when_merge[i] = BIC(
                np.concatenate((target_bin[:i], [target_bin[i]+target_bin[i+1]], target_bin[i+2:])),
                np.concatenate((control_bin[:i], [control_bin[i]+control_bin[i+1]], control_bin[i+2:])),
                lambda_ = lambda_
            )

        # to see if match the stop condition
        if np.min(BIC_when_merge) >= ini_BIC:
            break
        else:
            ini_BIC = np.min(BIC_when_merge)
            bin_to_merge = np.argmin(BIC_when_merge)
            target_bin = np.concatenate((target_bin[:bin_to_merge], [target_bin[bin_to_merge]+target_bin[bin_to_merge+1]], target_bin[bin_to_merge+2:]))
            control_bin = np.concatenate((control_bin[:bin_to_merge], [control_bin[bin_to_merge]+control_bin[bin_to_merge+1]], control_bin[bin_to_merge+2:]))
            start_bin = np.concatenate((start_bin[:bin_to_merge], [min(start_bin[bin_to_merge],start_bin[bin_to_merge+1])], start_bin[bin_to_merge+2:]))
            end_bin = np.concatenate((end_bin[:bin_to_merge], [max(end_bin[bin_to_merge],end_bin[bin_to_merge+1])], end_bin[bin_to_merge+2:]))
    return target_bin, control_bin, start_bin, end_bin

In [17]:
test_trg = [40, 38, 2, 2, 51, 49]
test_ctl = [4, 4, 2, 2, 4, 4]

In [18]:
BICseq2_segmentation(test_trg, test_ctl, lambda_=2)

(array([182]), array([20]), array([0]), array([5]))

In [19]:
BICseq2_segmentation(test_trg, test_ctl, lambda_=1)

(array([ 78,   4, 100]), array([8, 4, 8]), array([0, 2, 4]), array([1, 3, 5]))

In [79]:
# obs_dict, exp_dict = load_bin_dir('CNV/K562_WGS_CNV/bin')
# BICseq_segmentation(obs_dict['chr1'], exp_dict['chr1'], lambda_=3)

In [36]:
test_obs, test_exp = list(), list()
with open('Exps/example_bin/example_bin3/example_bin.bin', 'r') as f:
    for line in f.readlines()[1:]:
        start_, end_, obs_, exp_, log2ratio = line.strip().split()
        test_obs.append(float(obs_))
        test_exp.append(float(exp_))

In [37]:
BICseq2_segmentation(test_obs, test_exp, lambda_=1)

(array([   1.44017193,  297.04980337,    3.32692052, 1405.90770222]),
 array([  10.,  248.,   14., 1074.]),
 array([  0,  10, 258, 272]),
 array([   9,  257,  271, 1345]))

In [20]:
def BICseq_segmentation(target_bin, control_bin, lambda_ = 2):
    if len(target_bin) != len(control_bin):
        raise Exception('target and control should have same number of bins')
    
    ini_BIC = BIC(target_bin, control_bin, lambda_)
    start_bin, end_bin = np.arange(len(target_bin)), np.arange(len(target_bin))

    while(True):
        # if only has one bin then don't need to merge
        if len(target_bin) == 1 :
            break
        
        # calculate each posible merge's BIC in this iteration
        BIC_when_merge = np.zeros(len(target_bin)-1)
        for i in range(len(BIC_when_merge)):
            BIC_when_merge[i] = BIC(
                np.concatenate((target_bin[:i], [target_bin[i]+target_bin[i+1]], target_bin[i+2:])),
                np.concatenate((control_bin[:i], [control_bin[i]+control_bin[i+1]], control_bin[i+2:])),
                lambda_ = lambda_
            )

        # to see if match the stop condition
        if np.max(BIC_when_merge) >= ini_BIC:
            break
        else:
            ini_BIC = np.max(BIC_when_merge)
            bin_to_merge = np.argmax(BIC_when_merge)
            target_bin = np.concatenate((target_bin[:bin_to_merge], [target_bin[bin_to_merge]+target_bin[bin_to_merge+1]], target_bin[bin_to_merge+2:]))
            control_bin = np.concatenate((control_bin[:bin_to_merge], [control_bin[bin_to_merge]+control_bin[bin_to_merge+1]], control_bin[bin_to_merge+2:]))
            start_bin = np.concatenate((start_bin[:bin_to_merge], [min(start_bin[bin_to_merge],start_bin[bin_to_merge+1])], start_bin[bin_to_merge+2:]))
            end_bin = np.concatenate((end_bin[:bin_to_merge], [max(end_bin[bin_to_merge],end_bin[bin_to_merge+1])], end_bin[bin_to_merge+2:]))
    return target_bin, control_bin, start_bin, end_bin

In [29]:
BICseq_segmentation(test_trg, test_ctl, lambda_=1)

(array([182]), array([20]), array([0]), array([5]))